In [ ]:
from google.colab import files
import io, os, re, json, glob, shutil, subprocess, sys
import numpy as np
import pandas as pd

# ================= SETTINGS =================
name = "new_data"      # change if you rerun with a different dataset
BASE = "/content"
PY = sys.executable
N_SYN = 10000           # synthetic rows to generate
VAE_EPOCHS = 500
DIFF_EPOCHS = 1500
WANT = ["Tau_c", "PI_v", "T_va", "RSSC Vel", "CAV", "CAD", "ID_2", "IV_2",
        "Arias intensity UD", "PGV UD 3 sec", "PGD UD 3 sec", "Magnitude"]
TARGET = "Magnitude"
D = " --dataname " + name
os.chdir(BASE)


def run(cmd, cwd=None):
    print("\n$ " + cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, text=True, bufsize=1,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    if p.returncode != 0:
        print("!! failed, exit code", p.returncode)
    return p.returncode == 0


def rm(*paths):
    for p in paths:
        if os.path.isdir(p):
            shutil.rmtree(p, ignore_errors=True)
        elif os.path.exists(p):
            os.remove(p)


def patch_file(path, pattern, repl, label):
    t = open(path).read()
    t2 = re.sub(pattern, repl, t)
    open(path, "w").write(t2)
    print(label, "-> patched" if t != t2 else "-> unchanged/not found")


def prep(raw, label, impute=True):
    look = {re.sub(r"\s+", " ", c.strip()).lower(): c for c in raw.columns}
    miss = [w for w in WANT if w.lower() not in look]
    if miss:
        raise ValueError(label + " is missing columns: " + str(miss)
                         + "\nAvailable: " + str(list(raw.columns)))
    d = raw[[look[w.lower()] for w in WANT]].copy()
    d.columns = WANT
    d = d.apply(pd.to_numeric, errors="coerce")
    d = d.dropna(subset=[TARGET])
    d = d.fillna(d.median()) if impute else d.dropna()
    return d.reset_index(drop=True)


def fix_cols(s):
    if not all(c in s.columns for c in WANT):
        if s.shape[1] == len(WANT) + 1:
            print("Output columns are not named; assuming original order.")
            s = s.copy()
            s.columns = WANT + ["dummy_cat"]
        else:
            raise ValueError("Unexpected output columns: " + str(list(s.columns)))
    return s[WANT]


# ---------- data ----------
print("Upload the NEW training CSV:")
up = files.upload()
fname = list(up.keys())[0]
df = prep(pd.read_csv(io.BytesIO(up[fname])), "train data")
print("Using", fname, df.shape)
print("Magnitude range: %.2f - %.2f" % (df[TARGET].min(), df[TARGET].max()))

# TabSyn needs a categorical column: add a dummy, removed from the output
df["dummy_cat"] = np.random.RandomState(0).choice(["a", "b"], size=len(df))
cols = list(df.columns)
tidx = cols.index(TARGET)
info = {"name": name, "task_type": "regression", "header": "infer",
        "column_names": None,
        "num_col_idx": [i for i, c in enumerate(cols)
                        if i != tidx and c != "dummy_cat"],
        "cat_col_idx": [cols.index("dummy_cat")],
        "target_col_idx": [tidx], "file_type": "csv",
        "data_path": "data/%s/%s.csv" % (name, name), "test_path": None}

# ---------- TabSyn repo ----------
TS = BASE + "/tabsyn"
if not os.path.exists(TS):
    run("git clone https://github.com/amazon-science/tabsyn.git", cwd=BASE)
os.makedirs(TS + "/data/" + name, exist_ok=True)
os.makedirs(TS + "/data/Info", exist_ok=True)
df.to_csv(TS + "/data/%s/%s.csv" % (name, name), index=False)
with open(TS + "/data/Info/%s.json" % name, "w") as f:
    json.dump(info, f, indent=4)

req = []
for line in open(TS + "/requirements.txt").read().splitlines():
    b = re.split(r"[=<>~!]", line.strip())[0].strip().lower()
    if b.startswith("torch"):
        continue
    req.append(b if b in ("sdmetrics", "click") else line)
open(TS + "/requirements_no_torch.txt", "w").write("\n".join(req) + "\n")
if not run(PY + " -m pip install -r requirements_no_torch.txt -q", cwd=TS):
    extra = "icecream category_encoders tomli tomli_w zero scikit-learn"
    run(PY + " -m pip install " + extra + " einops tqdm sdmetrics -q", cwd=TS)
run(PY + " -m pip install sdmetrics tomli -q", cwd=TS)

for fp in glob.glob(TS + "/**/*.py", recursive=True):
    t = open(fp).read()
    if ", verbose=True)" in t:
        open(fp, "w").write(t.replace(", verbose=True)", ")"))
        print("Fixed verbose in", fp)
patch_file(TS + "/tabsyn/vae/main.py", r"NUM_EPOCHS\s*=\s*\d+",
           "NUM_EPOCHS = %d" % VAE_EPOCHS, "VAE epochs")
patch_file(TS + "/tabsyn/main.py", r"num_epochs\s*=\s*\d+(\s*\+\s*\d+)?",
           "num_epochs = %d" % DIFF_EPOCHS, "Diffusion epochs")
patch_file(TS + "/tabsyn/sample.py", r"num_samples\s*=\s*[\w\.\[\]]+",
           "num_samples = %d" % N_SYN, "Sample size")

# ---------- train + sample ----------
for p in glob.glob(TS + "/data/%s/*.npy" % name):
    rm(p)
rm(TS + "/tabsyn/vae/ckpt/" + name, TS + "/tabsyn/ckpt/" + name,
   TS + "/synthetic/" + name)

ok = run(PY + " process_dataset.py" + D, cwd=TS)
for step in (" --method vae --mode train", " --method tabsyn --mode train",
             " --method tabsyn --mode sample"):
    if ok:
        ok = run(PY + " main.py" + D + step, cwd=TS)

out = TS + "/synthetic/%s/tabsyn.csv" % name
if ok and os.path.exists(out):
    s = fix_cols(pd.read_csv(out))
    print("Generated rows:", len(s))
    if len(s) != N_SYN:
        s = s.sample(n=N_SYN, replace=len(s) < N_SYN, random_state=42)
    s = s.reset_index(drop=True)
    s.to_csv(BASE + "/tabsyn_synthetic.csv", index=False)
    print("Saved tabsyn_synthetic.csv", s.shape)
    print("Magnitude  real: %.2f-%.2f | synthetic: %.2f-%.2f"
          % (df[TARGET].min(), df[TARGET].max(),
             s[TARGET].min(), s[TARGET].max()))
    files.download(BASE + "/tabsyn_synthetic.csv")
else:
    print("TabSyn produced no output; see errors above.")

Upload the NEW training CSV:


Streaming output truncated to the last 5000 lines.
Epoch 669/1500: 100%|██████████| 1/1 [00:00<00:00,  5.43it/s, Loss=0.202]

Epoch 670/1500: 100%|██████████| 1/1 [00:00<00:00,  6.00it/s, Loss=0.209]

Epoch 671/1500: 100%|██████████| 1/1 [00:00<00:00,  5.92it/s, Loss=0.2]

Epoch 672/1500: 100%|██████████| 1/1 [00:00<00:00,  5.88it/s, Loss=0.199]

Epoch 673/1500: 100%|██████████| 1/1 [00:00<00:00,  5.91it/s, Loss=0.195]

Epoch 674/1500: 100%|██████████| 1/1 [00:00<00:00,  5.18it/s, Loss=0.201]

Epoch 675/1500: 100%|██████████| 1/1 [00:00<00:00,  5.12it/s, Loss=0.203]

Epoch 676/1500: 100%|██████████| 1/1 [00:00<00:00,  6.07it/s, Loss=0.205]

Epoch 677/1500: 100%|██████████| 1/1 [00:00<00:00,  5.71it/s, Loss=0.199]

Epoch 678/1500: 100%|██████████| 1/1 [00:00<00:00,  5.58it/s, Loss=0.205]

Epoch 679/1500: 100%|██████████| 1/1 [00:00<00:00,  5.77it/s, Loss=0.202]

Epoch 680/1500: 100%|██████████| 1/1 [00:00<00:00,  5.68it/s, Loss=0.201]

Epoch 681/1500: 100%|██████████| 1/1 [00:00<00:00, 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import time

STEPS = 5000        # training steps (repo default is much higher)
BATCH = 256
M = " --method tabrep_flow"
TR = BASE + "/TabRep"

src_csv = TS + "/data/%s/%s.csv" % (name, name)
src_info = TS + "/data/Info/%s.json" % name
if not (os.path.exists(src_csv) and os.path.exists(src_info)):
    raise FileNotFoundError("Run Cell 1 first (it prepares the data).")

if not os.path.exists(TR):
    run("git clone https://github.com/jacobyhsi/TabRep.git", cwd=BASE)

rp = TR + "/requirements.txt"
if os.path.exists(rp):
    req = []
    for line in open(rp).read().splitlines():
        b = re.split(r"[=<>~!]", line.strip())[0].strip().lower()
        if b.startswith("torch"):
            continue
        req.append(b if b in ("sdmetrics", "click") else line)
    open(TR + "/requirements_no_torch.txt", "w").write("\n".join(req) + "\n")
    run(PY + " -m pip install -r requirements_no_torch.txt -q", cwd=TR)
run(PY + " -m pip install tomli tomli_w icecream category_encoders -q", cwd=TR)
import tomli, tomli_w

for fp in glob.glob(TR + "/**/*.py", recursive=True):
    t = open(fp).read()
    if ", verbose=True)" in t:
        open(fp, "w").write(t.replace(", verbose=True)", ")"))

# ---- data (same prepared data as TabSyn) ----
os.makedirs(TR + "/data/" + name, exist_ok=True)
os.makedirs(TR + "/data/Info", exist_ok=True)
shutil.copy(src_csv, TR + "/data/%s/%s.csv" % (name, name))
shutil.copy(src_info, TR + "/data/Info/%s.json" % name)
for p in glob.glob(TR + "/data/%s/*.npy" % name):
    rm(p)
rm(TR + "/synthetic/" + name, TR + "/tabrep_flow/ckpt/" + name)

ok = run(PY + " process_dataset.py" + D, cwd=TR)
d_num = np.load(TR + "/data/%s/X_num_train.npy" % name,
                allow_pickle=True).shape[1]
print("numerical features:", d_num)

# ---- config built from the repo's regression example ----
cfg_path = TR + "/configs/%s.toml" % name
with open(TR + "/configs/beijing.toml", "rb") as f:
    cfg = tomli.load(f)


def sub(o):
    if isinstance(o, dict):
        return {k: sub(v) for k, v in o.items()}
    if isinstance(o, list):
        return [sub(v) for v in o]
    if isinstance(o, str):
        return o.replace("beijing", name)
    return o


cfg = sub(cfg)
cfg["num_numerical_features"] = d_num
cfg["task_type"] = "regression"
cfg.setdefault("train", {}).setdefault("main", {})
cfg["train"]["main"]["steps"] = STEPS
cfg["train"]["main"]["batch_size"] = BATCH
cfg.setdefault("sample", {"batch_size": 10000})
cfg["sample"]["num_samples"] = N_SYN
with open(cfg_path, "wb") as f:
    tomli_w.dump(cfg, f)
print("\n----- config -----")
print(open(cfg_path).read())

# ---- train + sample ----
t0 = time.time()
if ok:
    ok = run(PY + " main.py" + D + M + " --mode train", cwd=TR)
if ok:
    ok = run(PY + " main.py" + D + M + " --mode sample", cwd=TR)

outs = [p for p in glob.glob(TR + "/**/*.csv", recursive=True)
        if os.path.getmtime(p) > t0 and "/data/" not in p]
print("new csv files:", outs)
if ok and outs:
    src = sorted(outs, key=os.path.getmtime)[-1]
    print("reading", src)
    s = fix_cols(pd.read_csv(src))
    print("Generated rows:", len(s))
    if len(s) != N_SYN:
        s = s.sample(n=N_SYN, replace=len(s) < N_SYN, random_state=42)
    s = s.reset_index(drop=True)
    s.to_csv(BASE + "/tabrep_synthetic.csv", index=False)
    print("Saved tabrep_synthetic.csv", s.shape)
    print("Magnitude  real: %.2f-%.2f | synthetic: %.2f-%.2f"
          % (df[TARGET].min(), df[TARGET].max(),
             s[TARGET].min(), s[TARGET].max()))
    files.download(BASE + "/tabrep_synthetic.csv")
else:
    print("TabRep produced no output; see errors above.")

Streaming output truncated to the last 5000 lines.
Step 50/5000 Loss: 2.0155
Step 51/5000 Loss: 2.1316
Step 52/5000 Loss: 2.0741
Step 53/5000 Loss: 2.0161
Step 54/5000 Loss: 1.8719
Step 55/5000 Loss: 2.0064
Step 56/5000 Loss: 1.7707
Step 57/5000 Loss: 1.7853
Step 58/5000 Loss: 2.1835
Step 59/5000 Loss: 1.9302
Step 60/5000 Loss: 1.9359
Step 61/5000 Loss: 2.0801
Step 62/5000 Loss: 1.9158
Step 63/5000 Loss: 1.8868
Step 64/5000 Loss: 1.9785
Step 65/5000 Loss: 1.8791
Step 66/5000 Loss: 1.9628
Step 67/5000 Loss: 1.865
Step 68/5000 Loss: 1.7903
Step 69/5000 Loss: 1.9143
Step 70/5000 Loss: 2.0472
Step 71/5000 Loss: 1.7789
Step 72/5000 Loss: 1.7133
Step 73/5000 Loss: 1.6861
Step 74/5000 Loss: 1.6872
Step 75/5000 Loss: 1.7659
Step 76/5000 Loss: 1.679
Step 77/5000 Loss: 1.78
Step 78/5000 Loss: 1.7996
Step 79/5000 Loss: 1.7622
Step 80/5000 Loss: 1.8461
Step 81/5000 Loss: 1.8343
Step 82/5000 Loss: 1.9706
Step 83/5000 Loss: 1.8775
Step 84/5000 Loss: 1.8245
Step 85/5000 Loss: 1.6195
Step 86/5000 Loss

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import io, os, re, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from google.colab import files

WANT = ["Tau_c", "PI_v", "T_va", "RSSC Vel", "CAV", "CAD", "ID_2", "IV_2",
        "Arias intensity UD", "PGV UD 3 sec", "PGD UD 3 sec", "Magnitude"]
TARGET = "Magnitude"


def prep(raw, label):
    look = {re.sub(r"\s+", " ", c.strip()).lower(): c for c in raw.columns}
    miss = [w for w in WANT if w.lower() not in look]
    if miss:
        raise ValueError(label + " is missing columns: " + str(miss)
                         + "\nAvailable: " + str(list(raw.columns)))
    d = raw[[look[w.lower()] for w in WANT]].copy()
    d.columns = WANT
    return d.apply(pd.to_numeric, errors="coerce")


print("Select: tabsyn_synthetic.csv, tabrep_synthetic.csv, and your real "
      "data (original train and/or test csv)")
up = files.upload()
roles = {}
for fn, b in up.items():
    l = fn.lower()
    if "augment" in l:
        continue
    if "tabsyn" in l:
        roles["tabsyn"] = (fn, b)
    elif "tabrep" in l:
        roles["tabrep"] = (fn, b)
    elif "test" in l:
        roles["test"] = (fn, b)
    else:
        roles["orig"] = (fn, b)

ref_role = "orig" if "orig" in roles else "test"
if ref_role not in roles:
    raise FileNotFoundError("Upload the real data (train or test csv).")
print("\nReal reference:", roles[ref_role][0])

sets = {"Real": prep(pd.read_csv(io.BytesIO(roles[ref_role][1])), "real data")}
for key, label in (("tabsyn", "TabSyn"), ("tabrep", "TabRep")):
    if key in roles:
        sets[label] = prep(pd.read_csv(io.BytesIO(roles[key][1])), label)
        print(label, ":", roles[key][0], sets[label].shape)
if len(sets) < 2:
    raise FileNotFoundError("Upload at least one synthetic csv.")

corr = {k: v.corr(method="pearson") for k, v in sets.items()}
syn_names = [k for k in sets if k != "Real"]
iu = np.triu_indices(len(WANT), k=1)


def heat(ax, M, title, vmin, vmax, cmap):
    im = ax.imshow(M.values, vmin=vmin, vmax=vmax, cmap=cmap)
    ax.set_xticks(range(len(WANT)))
    ax.set_yticks(range(len(WANT)))
    ax.set_xticklabels(WANT, rotation=90, fontsize=8)
    ax.set_yticklabels(WANT, fontsize=8)
    for i in range(len(WANT)):
        for j in range(len(WANT)):
            v = M.values[i, j]
            ax.text(j, i, "%.2f" % v, ha="center", va="center", fontsize=6,
                    color="white" if abs(v) > 0.6 * max(abs(vmin), abs(vmax))
                    else "black")
    ax.set_title(title, fontsize=12)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)


# ---- 1. correlation heatmaps: Real, TabSyn, TabRep ----
fig, axes = plt.subplots(1, len(sets), figsize=(9 * len(sets), 8.5))
axes = np.atleast_1d(axes)
for ax, (k, M) in zip(axes, corr.items()):
    heat(ax, M, "Pearson correlation: " + k, -1, 1, "coolwarm")
plt.tight_layout()
plt.savefig("/content/p1_correlations.png", dpi=150)
plt.show()

# ---- 2. absolute difference from Real ----
fig, axes = plt.subplots(1, len(syn_names), figsize=(9 * len(syn_names), 8.5))
axes = np.atleast_1d(axes)
print("\nMean absolute difference in correlations vs Real (lower = better):")
for ax, k in zip(axes, syn_names):
    D = (corr[k] - corr["Real"]).abs()
    mad = D.values[iu].mean()
    print("  %-7s %.4f   (max single-pair gap: %.4f)"
          % (k, mad, D.values[iu].max()))
    heat(ax, D, "|%s - Real|   mean = %.4f" % (k, mad), 0, 0.5, "Reds")
plt.tight_layout()
plt.savefig("/content/p2_difference.png", dpi=150)
plt.show()

# ---- 3. correlation of each feature with Magnitude ----
feats = [c for c in WANT if c != TARGET]
x = np.arange(len(feats))
w = 0.8 / len(sets)
fig, ax = plt.subplots(figsize=(16, 6))
for i, (k, M) in enumerate(corr.items()):
    bars = ax.bar(x + i * w - 0.4 + w / 2, M.loc[feats, TARGET].values, w,
                  label=k)
    ax.bar_label(bars, fmt="%.2f", fontsize=7, rotation=90, padding=2)
ax.axhline(0, color="k", linewidth=1)
ax.set_xticks(x)
ax.set_xticklabels(feats, rotation=35, ha="right")
ax.set_ylabel("Pearson r with " + TARGET)
ax.set_title("Feature vs Magnitude correlation")
ax.margins(y=0.2)
ax.grid(axis="y", alpha=0.3)
ax.legend()
plt.tight_layout()
plt.savefig("/content/p3_magnitude_corr.png", dpi=150)
plt.show()

for f in ("p1_correlations", "p2_difference", "p3_magnitude_corr"):
    files.download("/content/%s.png" % f)

Select: tabsyn_synthetic.csv, tabrep_synthetic.csv, and your real data (original train and/or test csv)
